In [1]:
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error

In [24]:
Data = fetch_california_housing(as_frame=True) # Загрузим данные

X = Data['data']
y = Data['target']

In [3]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [4]:
y.head()

0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64

In [25]:
# Делим выборку на тренировочную (train) и тестовую (test)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train,y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)#random_state фиксирует разбиение датасета на test и train

In [81]:
# Обучение модели Линейной регрессии на тренировочных данных, предсказание и проверка качества
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

prediction = model.predict(X_test)
MSE = mean_squared_error(y_test, prediction)
RMSE = mean_squared_error(y_test, prediction) ** 0.5
print(MSE, RMSE)

0.5411287478470689 0.7356145375446769


In [54]:
# Обучение модели Лассо (регуляризация L1) на тренировочных данных, предсказание и проверка качества
from sklearn.linear_model import Lasso
model2 = Lasso(alpha=0.01)
model2.fit(X_train, y_train)

prediction2 = model2.predict(X_test)
MSE2 = mean_squared_error(y_test, prediction2)
RMSE2 = mean_squared_error(y_test, prediction2) ** 0.5
print(MSE2, RMSE2)

0.5320268525843376 0.7294017086519181


In [88]:
(model2.coef_ == 0) # посмотрел, занулил ли Лассо признаки

array([False, False, False, False, False, False, False, False])

In [ ]:
# Обучение модели Риджа (регуляризация L2) на тренировочных данных, предсказание и проверка качества
from sklearn.linear_model import Ridge
model1 = Ridge(alpha=1000)
model1.fit(X_train, y_train)

prediction1 = model1.predict(X_test)
MSE1 = mean_squared_error(y_test, prediction1)
RMSE1 = mean_squared_error(y_test, prediction1) ** 0.5
print(MSE1, RMSE1)

## ElacticNet

In [86]:
# Обучение модели ElasticNet (регуляризации L1 и L2) на тренировочных данных, предсказание и проверка качества
from sklearn.linear_model import ElasticNet
model3 = ElasticNet(alpha=0.0011008110571039537, l1_ratio=0.1)
model3.fit(X_train, y_train)

prediction3 = model3.predict(X_test)
MSE3 = mean_squared_error(y_test, prediction3)
RMSE3 = mean_squared_error(y_test, prediction3) ** 0.5
print(MSE3, RMSE3)

0.5397715660314378 0.7346914767652051


In [77]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1]
}

elastic = ElasticNet(max_iter=10000)
grid = GridSearchCV(elastic, param_grid, cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)
print(grid.best_params_)

{'alpha': 0.001, 'l1_ratio': 0.1}


In [83]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform

# Для alpha используем лог-равномерное распределение
param_dist = {
    'alpha': loguniform(1e-3, 1e3),
    'l1_ratio': uniform(0.1, 0.9)  # равномерно от 0.1 до 1 (shift+scale)
}

random_search = RandomizedSearchCV(
    elastic, param_dist, n_iter=100, cv=5, scoring='neg_mean_squared_error', 
    n_jobs=-1, random_state=42
)
random_search.fit(X_train, y_train)
print(random_search.best_params_)

{'alpha': np.float64(0.0011008110571039537), 'l1_ratio': np.float64(0.5596725723198092)}


In [85]:
from sklearn.linear_model import ElasticNetCV
model_cv = ElasticNetCV(alphas=[0.01, 0.1, 1.0, 10.0], 
                        l1_ratio=[0.1, 0.5, 0.7, 0.9, 1.0],
                        cv=5, random_state=42)
model_cv.fit(X_train, y_train)

print("Лучший alpha:", model_cv.alpha_)
print("Лучший l1_ratio:", model_cv.l1_ratio_)

Лучший alpha: 0.01
Лучший l1_ratio: 0.1


In [89]:
# Добавим полиномиальные признаки второй степени
# 1, x1, x2 -> 1, x1, x2, x1 * x2, x1 ** 2, x2 ** 2

from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree = 2)

pf.fit(X_train)

X_train_new = pf.transform(X_train)
X_test_new = pf.transform(X_test)

In [19]:
X_train[:1].shape

(1, 8)

In [90]:
model.fit(X_train_new, y_train)

prediction4 = model.predict(X_test_new)

MSE4 = mean_squared_error(y_test, prediction4)
RMSE4 = mean_squared_error(y_test, prediction4) ** 0.5

print(MSE4, RMSE4)

0.45478925099206124 0.6743806425098968


In [91]:
# Обучение модели Лассо (регуляризация L1) на тренировочных данных, предсказание и проверка качества
from sklearn.linear_model import Lasso
model2 = Lasso(alpha=0.01)
model2.fit(X_train_new, y_train)

prediction2 = model2.predict(X_test_new)
MSE2 = mean_squared_error(y_test, prediction2)
RMSE2 = mean_squared_error(y_test, prediction2) ** 0.5
print(MSE2, RMSE2)

0.46766103961498606 0.6838574702487252


C:\Users\1\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.640e+03, tolerance: 2.066e+00
  model = cd_fast.enet_coordinate_descent(
